In [1]:
import os
import re
import time
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy import stats
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import ContextualCompressionRetriever, EnsembleRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from dotenv import load_dotenv

load_dotenv()

MODEL = "gpt-4o-mini"

llm = ChatOpenAI(model=MODEL)
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [2]:
documents = [
    Document(page_content="트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.", metadata={"id": "d1"}),
    Document(page_content="BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.", metadata={"id": "d2"}),
    Document(page_content="GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.", metadata={"id": "d3"}),
    Document(page_content="RAG는 검색 증강 생성 기법으로 외부 지식을 LLM에 결합하여 할루시네이션을 줄입니다.", metadata={"id": "d4"}),
    Document(page_content="벡터 데이터베이스는 임베딩 벡터를 저장하고 유사도 기반 검색을 수행합니다. FAISS, Pinecone 등이 있습니다.", metadata={"id": "d5"}),
    Document(page_content="파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.", metadata={"id": "d6"}),
    Document(page_content="프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.", metadata={"id": "d7"}),
    Document(page_content="토큰화는 텍스트를 모델이 처리할 수 있는 단위로 분할하는 과정입니다. BPE, WordPiece 등이 사용됩니다.", metadata={"id": "d8"}),
]

In [3]:
#re-ranking
vectorstore = FAISS.from_documents(documents, embeddings_model)
bm_25_retriever = BM25Retriever.from_documents(documents, k=5)

In [4]:
doc_embeddings = {}

def get_embedding(text):
    return np.array(embeddings_model.embed_query(text))

for doc in doc_embeddings:
    doc_embeddings[doc.metadata['id']] = get_embedding(doc.page_content)

In [5]:
#re-ranking 필요한 이유 임베딩 검색 내에서 document에서 유사한 값을 가지오는데
#query와 doc의 상호관계
def vector_search(query, vectorstore, top_k=5):
    results = vectorstore.similarity_search_with_score(query, k=top_k)
    return [(doc, 1.0 / (1.0 + score)) for doc, score in results]

query = "트랜스포머와 BERT의 관계"
results = vector_search(query, vectorstore)
results

[(Document(id='f88fe4c4-d907-4ef1-a10b-1a5503f69045', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  np.float32(0.5249158)),
 (Document(id='cb7886fc-3cc5-4c6e-95e9-859d61451cce', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  np.float32(0.46390262)),
 (Document(id='7b1118db-04ca-43fe-a462-665427d31090', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  np.float32(0.41886568)),
 (Document(id='4d37ef62-f8f6-4f0f-9738-f68029e27a2c', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  np.float32(0.4128145)),
 (Document(id='c927e3a5-f2d9-4160-83fa-59ac08ca4937', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  np.float32(0.39726147))]

In [6]:
"""
import numpy as np

class AdaptiveFilter:
    @staticmethod
    def scored_docs(scored_docs):
        scores = [s for _, s in scored_docs]
        std = np.std(scores)
        score_range = max(scores) - min(scores)

        if std > 0.2:
            return ScoreFilter.score_gap(scored_docs, 2)
        elif score_range < 0.1:
            return ScoreFilter.fixed_threshold(scored_docs, 0.5)
        else:
            return ScoreFilter.dynamic_threshold(scored_docs, std_factor=1)


AdaptiveFilter.scored_docs(results)
"""

'\nimport numpy as np\n\nclass AdaptiveFilter:\n    @staticmethod\n    def scored_docs(scored_docs):\n        scores = [s for _, s in scored_docs]\n        std = np.std(scores)\n        score_range = max(scores) - min(scores)\n\n        if std > 0.2:\n            return ScoreFilter.score_gap(scored_docs, 2)\n        elif score_range < 0.1:\n            return ScoreFilter.fixed_threshold(scored_docs, 0.5)\n        else:\n            return ScoreFilter.dynamic_threshold(scored_docs, std_factor=1)\n\n\nAdaptiveFilter.scored_docs(results)\n'

In [7]:
document = "BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다. 트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다. GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다."

In [8]:
# 하네스 엔지니어링 : compress 전략 100k -> 압축해라, 턴이 20턴정도 끝나면 압축해라
# context compression
# retriever : Embedding, Bm25, Hybrid, MMR
# MMR(Maximun Marginal Relevance)

# 정규표현식
import re

def extractive_compress(query, document, max_sentences=3):
    sentences = re.split(r'[.!?]\s*', document)

    if not sentences:
        return document

    query_terms = set(query.lower().split())
    scored = []

    for sent in sentences:
        sent_terms = set(sent.lower().split())
        overlap = len(query_terms & sent_terms)
        scored.append((sent, overlap))

    # 점수 기준 정렬
    scored.sort(key=lambda x: x[1], reverse=True)

    # 상위 문장 선택
    selected = [s for s, _ in scored[:max_sentences] if s.strip()]

    return ". ".join(selected) + "."

In [9]:
long_doc = "트랜스포머는 2017년 구글이 발표한 아키텍처입니다. Self-Attention 메커니즘이 핵심입니다. 이전의 RNN, LSTM과 달리 병렬 처리가 가능합니다. BERT와 GPT 모두 트랜스포머를 기반으로 합니다. 자연어 처리뿐 아니라 컴퓨터 비전에서도 활용됩니다"
query = "트랜스포머와 BERT의 관계"

compressed = extractive_compress(query, long_doc, max_sentences=3)
compressed

'트랜스포머는 2017년 구글이 발표한 아키텍처입니다. Self-Attention 메커니즘이 핵심입니다. 이전의 RNN, LSTM과 달리 병렬 처리가 가능합니다.'

In [10]:
compress_chain = ChatPromptTemplate.from_messages([
    ("system", "당신은 문서 요약 전문가입니다."),
    ("human", """다음 문서에서 쿼리에 답하는데 필요한 핵심 정보만 추출하세요.
    불필요한 내용은 제거하고, {max_tokens}자 이내로 압축하세요.

    쿼리 : {query}
    문서 : {document}

    압축결과 :
    """)
]) | llm | StrOutputParser()

In [11]:
def llm_compress(query, document, max_tokens=100):
    return compress_chain.invoke({
        'query' : query,
        'document' : document,
        'max_tokens' : max_tokens
    })

In [12]:
compressed_llm = llm_compress(query, long_doc)

In [13]:
compressed_llm

'BERT는 트랜스포머 아키텍처를 기반으로 하며, 2017년 구글이 발표했습니다.'

In [14]:
compressor = LLMChainExtractor.from_llm(llm)
compressor_retriever = ContextualCompressionRetriever(base_compressor=compressor,
                                                      base_retriever=vectorstore.as_retriever(search_kwargs={'k' : 3}))

In [15]:
compressed_docs = compressor_retriever.invoke(query)
compressed_docs

[Document(metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
 Document(metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.')]

In [16]:
# 압축 품질 평가 : 키워드 보존유르, 의미 보존도
def evaluate_compression(original, compressed, query):
    # 1. 압축율
    ratio = len(compressed) / len(original)

    # 2. 키워드 보존율
    query_terms = set(query.lower().split())
    orig_terms = set(original.lower().split())
    comp_terms = set(compressed.lower().split())

    orig_query_terms = query_terms & orig_terms
    comp_query_terms = query_terms & comp_terms
    keyword_covarage = len(comp_query_terms) / len(orig_query_terms) if orig_query_terms else 1.0

    # 3. 의미 보존도
    original_emb = np.array(embeddings_model.embed_query(original))
    compressed_emb = get_embedding(compressed)

    senmantic_similarity = np.dot(original_emb, compressed_emb) / (np.linalg.norm(original_emb) * np.linalg.norm(compressed_emb))
    
    return {
        'compression_ratio' : ratio,
        'keyword_covarage' : keyword_covarage,
        'senmantic_similarity' : senmantic_similarity
    }

In [17]:
evaluate_compression(long_doc, compressed_llm, query)

{'compression_ratio': 0.29931972789115646,
 'keyword_covarage': 1.0,
 'senmantic_similarity': np.float64(0.5986194891286959)}

In [18]:
def cosine_sim(original_emb, compressed_emb):
    return np.dot(original_emb, compressed_emb) / (np.linalg.norm(original_emb) * np.linalg.norm(compressed_emb))

In [19]:
import re

def mmr_compress(query, document, max_sentences=3, param=0.5):
    sentences = re.split(r'[.!?]\s*', document)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]

    if len(sentences) <= max_sentences:
        return '. '.join(sentences) + '.'

    query_emb = get_embedding(query)
    sent_emb = [get_embedding(s) for s in sentences]

    selected = []
    remaining = list(range(len(sentences)))

    for _ in range(max_sentences):
        best_score = -float('inf')
        best_idx = -1

        for idx in remaining:
            relevance = cosine_sim(query_emb, sent_emb[idx])

            if selected:
                max_sim = max(cosine_sim(sent_emb[idx], sent_emb[s]) for s in selected)
            else:
                max_sim = 0.0

            mmr = param * relevance - (1 - param) * max_sim

            if mmr > best_score:
                best_score = mmr
                best_idx = idx

        selected.append(best_idx)
        remaining.remove(best_idx)

    return '. '.join(sentences[i] for i in sorted(selected)) + '.'

In [20]:
mmr_compress(query, long_doc, max_sentences=2, param=0.7)

'트랜스포머는 2017년 구글이 발표한 아키텍처입니다. BERT와 GPT 모두 트랜스포머를 기반으로 합니다.'

In [21]:
def average_precision(retrieved_ids, relevant_ids):
    relevant_set = set(relevant_ids)
    hits = 0
    sum_precision = 0.0

    for i, doc_id in enumerate(retrieved_ids):
        if doc_id in relevant_set:
            hits += 1
            precision_at_i = hits / (i + 1)
            sum_precision += precision_at_i

    return sum_precision / len(relevant_set) if relevant_set else 0.0

In [22]:
def mean_average_precision(query_results):
    # query_results : {query : (retrieved_ids, relevant_ids)}
    aps = []
    for query, (retrieved, relevant) in query_results.items():
        ap = average_precision(retrieved, relevant)
        aps.append(ap)

    map_score = np.mean(aps)
    return map_score

In [23]:
relevant = {'d1', 'd2', 'd3'}
orig_order = [doc.metadata['id'] for doc, _ in results]
orig_order

['d1', 'd2', 'd3', 'd6', 'd7']

In [24]:
def intra_list_similarity(doc_ids, doc_embeddings, top_k=5):
    ids = doc_ids[:top_k]
    embs = [doc_embeddings[did] for did in ids if did in doc_embeddings]

    if len(embs) < 2:
        return 0.0

    sims = []
    for i in range(len(embs)):
        for j in range(i+1, len(embs)):
            sims.append(cosine_sim(embs[i], embs[j]))

    return np.mean(sims)

In [25]:
ils_score = intra_list_similarity(orig_order, doc_embeddings)
ils_score

0.0

In [26]:
class BM25Reranker:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b

    def rerank(self, query, search_results):
        docs = [doc for doc, _ in search_results]
        tokenized = [doc.page_content.lower().split() for doc in docs]

        avg_dl = np.mean([len(t) for t in tokenized])

        df_count = Counter()
        for tokens in tokenized:
            for t in set(tokens):
                df_count[t] += 1
        N = len(docs)
        
        query_tokens = query.lower().split()
        scored = []

        for i, doc in enumerate(docs):
            score = 0.0
            doc_len = len(tokenized[i])
            tf_counter = Counter(tokenized[i])

            for qt in query_tokens:
                tf = tf_counter.get(qt, 0)
                if tf == 0:
                    continue
                df = df_count.get(qt, 0)
                idf = math.log((N-df + 0.5) / (df + 0.5) + 1)
                numerator = tf * (self.k1 + 1)
                denominator = tf + self.k1 * (1 - self.b + self.b * doc_len /avg_dl)
                score +=  idf * numerator / denominator

            scored.append((doc, score))

        scored.sort(key = lambda x: x[1], reverse=True)
        return scored

In [27]:
# A/B test
def ab_test(scores_a, scores_b, alpha=0.05):
    t_stat, p_val = stats.ttest_rel(scores_a, scores_b)
    
    mean_a = np.mean(scores_a)
    mean_b = np.mean(scores_b)

    pooled_std = np.sqrt((np.std(scores_a)**2 + np.std(scores_b)**2)/2)
    cohens_d = (mean_a - mean_b) / pooled_std if pooled_std > 0 else 0.0

    return {
        'mean_A' : mean_a,
        'meab_B' : mean_b,
        'p_val' : p_val,
        'cohens_d' : cohens_d,
        'significant' : p_val < alpha
    }

In [28]:
test_queries = {
    '트랜스포머 아키텍쳐' : ({'d1, d2, d3'}),
    '벡터 검색 방법' : ({'d5'}),
    'LLM 학습 방법' : ({'d6, d7'}),
}

bm25_reranker = BM25Reranker()
aps_before = []
aps_after = []
for q, relevant in test_queries.items():
    orig = vector_search(q, vectorstore)
    reranked = bm25_reranker.rerank(q, orig)

    orig_ids = [doc.metadata['id'] for doc, _ in orig]
    reranked_ids = [doc.metadata['id'] for doc, _ in reranked]

    aps_before.append(average_precision(orig_ids, relevant))
    aps_after.append(average_precision(reranked_ids, relevant))

In [29]:
aps_before

[0.0, 1.0, 0.0]

In [30]:
result = ab_test(aps_before, aps_after, alpha=0.05)
result

{'mean_A': np.float64(0.3333333333333333),
 'meab_B': np.float64(0.16666666666666666),
 'p_val': np.float64(0.42264973081037427),
 'cohens_d': np.float64(0.4472135954999579),
 'significant': np.False_}

In [31]:
# LLM-as-judge
def basic_judge(question, answer, scale='1~5'):
    """기본 LLM 평가 : 질문-답변 쌍을 점수화"""
    prompt = f"""다음 답변을 {scale}스케일로 평가하세요.

    질문 : {question}
    답변 : {answer}

    평가 기준
    - 정확성 : 사실적으로 올바른가?
    - 완전성 : 질문에 충분히 답했는가?
    - 명확성 : 이해하기 쉽게 설명했는가?

    반드시 아래 형식으로 답하세요:
    점수 : [숫자]
    이유 : [한 줄 설명]
    """
    result = llm.invoke(prompt).content
    return result

In [32]:
question = "파이썬의 장점은 무엇인가요?"
answers = [
    "파이썬은 간결한 문법으로 초보자도 쉽게 배울 수 있으며, 풍부한 라이브러리와 활발한 커뮤니티가 있습니다.",
    "파이썬 좋아요.",
    "자바스크립트는 웹 개발에 주로 사용됩니다.",
]

In [33]:
for i, ans in enumerate(answers):
    result = basic_judge(question, ans)
    print(f"평가 : {result}")

평가 : 점수 : 5  
이유 : 파이썬의 장점에 대해 정확하고 간결하게 설명하며, 중요한 요소들을 모두 포함하고 있다.
평가 : 점수 : 1  
이유 : 답변이 매우 간단하고 질문에 대한 원래의 정보나 구체적인 장점이 전혀 제공되지 않았습니다.
평가 : 점수 : 1  
이유 : 답변이 질문에 전혀 관련이 없으며, 파이썬의 장점에 대한 내용을 포함하지 않습니다.


In [34]:
def simple_rouge_f1(reference, candidate):
    ref_words = Counter(reference.split())
    cand_words = Counter(candidate.split())
    overlap = sum(min(ref_words[w], cand_words.get(w, 0)) for w in ref_words)
    r = overlap / sum(ref_words.values()) if ref_words else 0
    p = overlap / sum(cand_words.values()) if cand_words else 0
    return 2*p*r / (p+r) if (p+r) > 0 else 0

In [35]:
import re

def basic_judge2(question, answer, scale='1~5'):
    prompt = f"""다음 답변을 {scale}스케일로 평가하세요.

질문 : {question}
답변 : {answer}

평가 기준
- 정확성 : 사실적으로 올바른가?
- 완전성 : 질문에 충분히 답했는가?
- 명확성 : 이해하기 쉽게 설명했는가?

반드시 아래 형식으로 답하세요:
점수 : [숫자]
신뢰도 : [0~1 사이 실수]
이유 : [한 줄 설명]
"""

    result = llm.invoke(prompt).content

    score = re.search(r"점수\s*:\s*(\d+)", result)
    conf = re.search(r"신뢰도\s*:\s*([0-9.]+)", result)

    return {
        "score": int(score.group(1)) if score else 0,
        "confidence": float(conf.group(1)) if conf else 0.0,
        "raw": result
    }

In [36]:
question = "딥러닝이란?"
reference = "딥러닝은 다층 신경망을 사용하여 데이터에서 복잡한 패턴을 학습하는 머신러닝 방법입니다"
test_answers = [
    ("딥러닝은 다층 신경망을 사용하여 데이터에서 복잡한 패턴을 학습하는 머신러닝 방법입니다", "정답 복사"),
    ("인공 신경망의 층을 깊게 쌓아 복잡한 문제를 풀 수 있는 AI 기술이에요", "좋은 패러프레이즈"),
    ("딥러닝 딥러닝 신경망 데이터 학습 패턴 머신러닝", "키워드 나열"),
]

rows = []
for ans, label in test_answers:
    rouge = simple_rouge_f1(reference, ans)
    llm_result = basic_judge2(question, ans)

    rows.append({
        '유형': label,
        'rouge': rouge,
        'llm_score': llm_result['score'],
        'llm_confidence': llm_result['confidence'],
    })

In [37]:
rows

[{'유형': '정답 복사', 'rouge': 1.0, 'llm_score': 4, 'llm_confidence': 0.9},
 {'유형': '좋은 패러프레이즈',
  'rouge': 0.0909090909090909,
  'llm_score': 3,
  'llm_confidence': 0.7},
 {'유형': '키워드 나열',
  'rouge': 0.11764705882352941,
  'llm_score': 2,
  'llm_confidence': 0.3}]

In [38]:
RUBRIC = {
    5: "정확하고 완전하며, 예시와 설명이 풍부하다",
    4: "정확하고 핵심을 다루지만, 일부 세부사항이 부족하다",
    3: "대체로 정확하지만, 중요한 내용 일부가 누락되었다",
    2: "부분적으로만 정확하거나, 핵심을 빗나갔다",
    1: "부정확하거나 질문과 무관하다",
}

def pointwise_judge(question, answer, rubric = RUBRIC):
    rubric_text = '\n'.join(
        f'{k} 점 : {v}' for k,v in sorted(rubric.items(), reverse=True)
    )

    prompt = f"""다음 답변을 아래 루브릭에 따라 평가해주세요.

    질문 : {question}
    답변 : {answer}

    루브릭 : {rubric_text}

    반드시 JSON으로 답하세요.
    {{"score":1-5, "reasoning":"이유"}}
    """

    result = llm.invoke(prompt).content
    result = result.replace('```json', '').replace('```', '')
    return json.loads(result)

In [39]:
question = "REST API와 GraphQL의 차이점을 설명해주세요"
answers = [
    "REST는 리소스 단위 URL에 HTTP 메서드를 사용하고, GraphQL은 단일 엔드포인트에서 클라이언트가 필요한 데이터를 쿼리합니다. REST는 오버/언더 페칭 문제가 있고, GraphQL은 이를 해결하지만 캐싱이 어렵습니다.",
    "REST는 URL을 사용하고 GraphQL은 쿼리를 사용합니다.",
    "둘 다 API입니다.",
]

for i, ans in enumerate(answers):
    result = pointwise_judge(question, ans)
    print(f"답변 : {i+1} : {ans}")
    print(f"점수 {result.get('score')}/5")

답변 : 1 : REST는 리소스 단위 URL에 HTTP 메서드를 사용하고, GraphQL은 단일 엔드포인트에서 클라이언트가 필요한 데이터를 쿼리합니다. REST는 오버/언더 페칭 문제가 있고, GraphQL은 이를 해결하지만 캐싱이 어렵습니다.
점수 4/5
답변 : 2 : REST는 URL을 사용하고 GraphQL은 쿼리를 사용합니다.
점수 2/5
답변 : 3 : 둘 다 API입니다.
점수 1/5


In [40]:
def pairwise_judge(question, answer_a, answer_b):
    prompt = f"""두 답변을 비교하여 더 나은족 선택하세요
    질문 : {question}
    답변 A : {answer_a}
    답변 B : {answer_b}

    반드시 JSON으로 답하세요.
    {{"winner" : "A 또는 B 또는 Tie", "reasoning" : "선택이유"}}
    """

    result = llm.invoke(prompt).content
    result = result.replace('```json', '').replace('```', '')
    return json.loads(result)

In [41]:
question = "마이크로서비스 아키텍처란?"
answer_a = "마이크로서비스는 애플리케이션을 작은 독립 서비스로 분리하는 아키텍처 패턴입니다. 각 서비스는 독립 배포, 확장이 가능합니다."
answer_b = "여러 개의 작은 서비스로 나누는 것입니다."

In [42]:
pairwise_judge(question, answer_a, answer_b)

{'winner': 'A',
 'reasoning': '답변 A는 마이크로서비스 아키텍처의 정의를 더 구체적으로 설명하고 있으며, 독립 배포와 확장의 가능성을 언급하여 이 패턴의 주요 특성을 강조하고 있습니다. 반면, 답변 B는 너무 간단하여 내용을 충분히 전달하지 못합니다.'}

In [43]:
def reference_based_judge(question, reference, answer):
    prompt = f"""당신은 AI 교육 전문가입니다.
    질문 : {question}
    정답 : {reference}
    학생 답변 : {answer}

    평가 기준:
    - 정답의 핵심 포인트를 얼마나 포함하는가
    - 사실적으로 틀린 내용이 있는가
    - 정답에 없는 올바른 추가 정보가 있는가
    
    반드시 JSON으로 답하세요.
    {{"score" : 1-5, "covered_points" : ["포함된 핵심 포인트들"], "missing_points":["누락된 포인트"], "errors":["틀린 내용"]}}
    """

    result = llm.invoke(prompt).content
    result = result.replace('```json', '').replace('```', '')
    return json.loads(result)

In [44]:
question = "TCP와 UDP의 차이점은?"
reference = "TCP는 연결지향적이고 신뢰성 있는 전송을 보장하며, UDP는 비연결지향적이고 빠르지만 신뢰성을 보장하지 않습니다."
answer = "TCP는 연결을 먼저 수립하고 데이터 전송의 순서와 무결성을 보장합니다. 반면 UDP는 연결 없이 바로 전송하여 빠르지만 패킷 손실이 가능합니다."

In [45]:
result = reference_based_judge(question, reference, answer)
result

{'score': 5,
 'covered_points': ['TCP는 연결을 먼저 수립한다.',
  'TCP는 데이터 전송의 순서와 무결성을 보장한다.',
  'UDP는 연결 없이 바로 전송한다.',
  'UDP는 빠르지만 패킷 손실이 가능하다.'],
 'missing_points': [],
 'errors': []}

In [46]:
def cot_judge(question, answer):
    prompt = f"""다음 답변을 단계별로 분석하세요.
    질문 : {question}
    답변 : {answer}

    step1: 질문이 요구하는 핵심 포인트를 나열하세요
    step2: 답변이 각 포인트를 다루는지 확인하세요
    step3: 답변의 강점과 약점을 정리하세요
    step4: 종합점수(1~5)를 부여하세요

    최종 JSON:
    {{"steps" : ["step1 결과", "step2 결과"], "score" : 1-5}}
    """

    result = llm.invoke(prompt).content
    result = result.replace('```json', '').replace('```', '')
    return json.loads(result)

In [47]:
question = "객체지향 프로그래밍의 4대 원칙을 설명해주세요"
answer = "캡슐화, 상속, 다형성이 있습니다. 캡슐화는 데이터를 숨기는 것이고 상속은 부모 클래스를 물려받는 것입니다."

cot = cot_judge(question, answer)

In [48]:
cot

{'steps': ['객체지향 프로그래밍의 4대 원칙: 캡슐화, 상속, 다형성',
  '답변은 캡슐화와 상속에 대한 설명만 포함되어 있고, 다형성에 대한 언급이 없으며, 각 개념에 대한 구체적인 설명 부족'],
 'score': 2}

In [49]:
import json

def fewshot_judge(question, answer):
    prompt = f"""
당신은 질문과 답변이 적절한지 판단하는 평가자입니다.
다음 기준으로 평가하세요:
- 질문에 대해 답변이 정확한가?
- 답변이 충분한가?

출력은 반드시 JSON 형식으로 하세요:
{{
  "score": 1~5 사이 정수,
  "reason": "평가 이유"
}}

Q: {question}
A: {answer}
결과:
"""

    result = llm.invoke(prompt).content
    result = result.replace('```json', '').replace('```', '')
    return json.loads(result)

In [50]:
question = "Git이란 무엇인가요?"
answer = "코드 버전 관리 도구입니다."

print(fewshot_judge(question, answer))

{'score': 3, 'reason': '답변은 Git의 기본적인 정의를 제공하지만, 무엇을 하는지, 어떻게 사용되는지에 대한 추가적인 설명이 부족하여 충분하지 않습니다.'}
